In [8]:
import re, zipfile, shutil
from pathlib import Path
import pandas as pd

In [9]:
BASE     = Path(r"C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS")
ZIP_DIR  = BASE  # ZIP 파일들이 있는 위치 (하위 폴더면 수정)
OUT_DIR  = BASE / "GTFS_CT"

CATEGORY_MAP = {
    "MA-BA"   : "행정경계",
    "MR-AML"  : "철도망",
    "MR-LLV2" : "도로망",
    "PT-RW"   : "철도",
    "PT-GTFS" : "대중교통GTFS",
}

YEAR_RE = re.compile(r"(\d{4})")

In [10]:
def detect_category(name: str) -> str:
    for code, label in CATEGORY_MAP.items():
        if code in name:
            return label
    return "기타"

def detect_year(name: str) -> str:
    m = YEAR_RE.search(name)
    return m.group(1) if m else "unknown"

def extract_recursive(zip_path: Path, dest: Path, depth: int = 0):
    if depth > 5:
        return
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            for info in zf.infolist():
                # 한글 파일명 CP949 디코딩
                try:
                    info.filename = info.filename.encode("cp437").decode("cp949")
                except (UnicodeDecodeError, UnicodeEncodeError):
                    pass  # 이미 UTF-8이거나 변환 불가면 원본 유지
                zf.extract(info, dest)
    except Exception as e:
        print(f"  [ERR] {zip_path.name}: {e}")
        return
    for nested in list(dest.rglob("*.zip")):
        extract_recursive(nested, nested.parent / nested.stem, depth + 1)
        nested.unlink()

In [11]:
zip_files = sorted(ZIP_DIR.glob("*.zip"))
print(f"ZIP 파일: {len(zip_files)}개\n")

manifest = []

for zp in zip_files:
    category = detect_category(zp.name)
    year     = detect_year(zp.name)
    dest     = OUT_DIR / category / year

    if dest.exists():
        print(f"[SKIP] {category}/{year}  ({zp.name})")
    else:
        extract_recursive(zp, dest)
        print(f"[OK]   {category}/{year}  ({zp.name})")

    manifest.append({"category": category, "year": year, "zip": zp.name})

print(f"\n완료")

ZIP 파일: 76개

[OK]   행정경계/2016  (2016-TM-GR-MA-BA 행정경계(2015년 기준).zip)
[OK]   철도망/2016  (2016-TM-GR-MR-AML 철도망(2015년 기준).zip)
[OK]   도로망/2016  (2016-TM-GR-MR-LLV2 도로망(2015년 기준).zip)
[SKIP] 행정경계/2016  (2016-TM-KA-MA-BA 행정경계(2015년 기준)_170518.zip)
[SKIP] 철도망/2016  (2016-TM-KA-MR-AML 철도망(2015년 기준).zip)
[SKIP] 도로망/2016  (2016-TM-KA-MR-LLV2 도로망(2015년 기준)_170929.zip)
[OK]   철도/2016  (2016-TM-PT-RW 철도(2015년 기준).zip)
[OK]   행정경계/2017  (2017-TM-GR-MA-BA 행정경계(2016년 기준).zip)
[OK]   철도망/2017  (2017-TM-GR-MR-AML 철도망(2016년 기준).zip)
[OK]   도로망/2017  (2017-TM-GR-MR-LLV2 도로망(2016년 기준).zip)
[SKIP] 행정경계/2017  (2017-TM-KA-MA-BA 행정경계(2016년 기준).zip)
[SKIP] 철도망/2017  (2017-TM-KA-MR-AML 철도망(2016년 기준).zip)
[SKIP] 도로망/2017  (2017-TM-KA-MR-LLV2 도로망(2016년 기준).zip)
[OK]   철도/2017  (2017-TM-PT-RW 철도(2016년 기준).zip)
[OK]   행정경계/2018  (2018-TM-GR-MA-BA 행정경계(2017년기준_세계좌표).zip)
[OK]   철도망/2018  (2018-TM-GR-MR-AML 철도망(2017년 기준).zip)
[OK]   도로망/2018  (2018-TM-GR-MR-LLV2 도로망(2017년 기준).zip)
[SKIP] 행정경계/2018  (2018-TM-KA-MA-BA 

In [12]:
# 결과 요약
df = pd.DataFrame(manifest).sort_values(["category", "year"]).reset_index(drop=True)
df.to_csv(OUT_DIR / "manifest.csv", index=False, encoding="utf-8-sig")

print("=== 카테고리 × 연도 ===")
print(pd.crosstab(df["category"], df["year"]))

=== 카테고리 × 연도 ===
year      2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
category                                                            
대중교통GTFS     0     0     0     0     0     0     1     1     2     2
도로망          2     2     2     2     2     2     2     2     4     0
철도           1     1     1     1     1     1     1     1     2     0
철도망          2     2     2     2     2     2     2     2     4     0
행정경계         2     2     2     2     2     2     2     2     4     0


In [13]:
# 전체 파일 목록 (확장자별 확인용)
records = []
for f in OUT_DIR.rglob("*"):
    if not f.is_file() or f.name == "manifest.csv":
        continue
    parts = f.relative_to(OUT_DIR).parts
    records.append({
        "category" : parts[0] if len(parts) > 0 else "",
        "year"     : parts[1] if len(parts) > 1 else "",
        "filename" : f.name,
        "ext"      : f.suffix.lower(),
        "size_kb"  : round(f.stat().st_size / 1024, 1),
    })

file_df = pd.DataFrame(records)
file_df.to_csv(OUT_DIR / "file_catalog.csv", index=False, encoding="utf-8-sig")

print("=== 확장자 분포 ===")
print(file_df.groupby("ext").size().sort_values(ascending=False).to_string())
print(f"\n전체 파일: {len(file_df)}개")

=== 확장자 분포 ===
ext
.dbf     89
.shx     89
.shp     89
.prj     81
.hwp     36
.txt     25
.sbx     18
.sbn     18
.xml      9
.xlsx     5
.pdf      4

전체 파일: 463개


In [14]:
# 카테고리별 파일 샘플
for cat, grp in file_df.groupby("category"):
    print(f"\n[{cat}] 총 {len(grp)}개")
    print(grp[["year", "filename", "ext", "size_kb"]].head(5).to_string(index=False))


[대중교통GTFS] 총 34개
year                   filename   ext  size_kb
2022 GTFS_202103_route세부정보.xlsx .xlsx   1284.1
2022        GTFS_202103_설명서.pdf  .pdf    196.3
2023 202203_GTFS_route세부정보.xlsx .xlsx   1297.7
2023  202203_GTFS_도시철도환승정보.xlsx .xlsx     18.0
2023        202203_GTFS_설명서.pdf  .pdf    218.6

[도로망] 총 144개
year                       filename  ext  size_kb
2016          2016년-GIS DB 설명자료.hwp .hwp    208.0
2017          2017년-GIS DB 설명자료.hwp .hwp    202.5
2018        2017년기준_GIS DB 설명자료.hwp .hwp  11920.0
2019        2018년기준_GIS DB 설명자료.hwp .hwp  12112.0
2020 2019년기준_GIS DB 설명자료_210429.hwp .hwp  12048.5

[철도] 총 145개
year                       filename  ext  size_kb
2016          2016년-GIS DB 설명자료.hwp .hwp    208.0
2017          2017년-GIS DB 설명자료.hwp .hwp    202.5
2018        2017년기준_GIS DB 설명자료.hwp .hwp  11920.0
2019        2018년기준_GIS DB 설명자료.hwp .hwp  12112.0
2020 2019년기준_GIS DB 설명자료_210429.hwp .hwp  12048.5

[철도망] 총 89개
year                       filename  ext  size_kb
2016      